# 06 -- Parameter stability

Paired script: `analysis/walk_forward.py` (same script as notebook 07, different framing).
This notebook looks at **stability**: how much does the test-period win rate/expectancy
vary window-to-window? A strategy whose out-of-sample performance swings wildly between
adjacent windows is a red flag regardless of its average performance.

**Uses clearly-labelled SYNTHETIC trade data.** Real-data run: PENDING.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.walk_forward import run

In [ ]:
# Same 5-trade fixture hand-verified in tests/test_walk_forward.py, spanning
# day 0 to day 8 -- 3 windows result from train_days=3/test_days=2/step_days=2.
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_stability_demo_"))

def row(trade_id, day_offset, exit_price, profit):
    exit_time = pd.Timestamp("2026-01-01", tz="UTC") + pd.Timedelta(days=day_offset)
    return {"trade_id": trade_id, "symbol": "XAUUSD", "is_long": "True",
            "entry_time": (exit_time - pd.Timedelta(hours=1)).isoformat(),
            "exit_time": exit_time.isoformat(), "entry_price": 100.0,
            "exit_price": exit_price, "stop_price": 98.0, "profit": profit}

pd.DataFrame([
    row("t0", 0, 104.0, 10.0), row("t1", 1, 99.0, -5.0), row("t2", 2, 103.0, 10.0),
    row("t4", 4, 105.0, 20.0), row("t8", 8, 99.0, -5.0),
]).to_csv(tmp_dir / "trades.csv", index=False)

In [ ]:
windows_df = run(tmp_dir / "trades.csv", train_days=3, test_days=2, step_days=2,
                  summary_json=tmp_dir / "summary.json", repo_path=PROJECT_ROOT.parents[1])

print(windows_df[["window_index", "train_n", "train_win_rate", "test_n", "test_expectancy_r"]])

test_expectancies = windows_df["test_expectancy_r"].dropna()
print(f"\nwindows with test data : {len(test_expectancies)} / {len(windows_df)}")
print(f"test expectancy spread : min={test_expectancies.min():.2f} max={test_expectancies.max():.2f}")

assert len(windows_df) == 3

## Real-data run: PENDING

A meaningful stability read needs many more windows than this 5-trade demo can provide -- requires real trade history spanning multiple months, which does not exist yet.